In [1]:
import requests
import pandas as pd
import numpy as np
import json
import janitor
import math
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [2]:
def setmyplotstyle():
    #plt.style.use('fivethirtyeight')
    plt.style.use('seaborn-v0_8-pastel')
    #plt.style.use('dark_background')
    plt.rcParams['figure.figsize'] = (22, 8)
    plt.rcParams['font.size'] = 14
    #plt.rcParams['font.family'] = 'sans-serif'
    #plt.rcParams['font.sans-serif'] = 'Helvetica'
    plt.rcParams['axes.facecolor'] = 'white'
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.grid'] = False
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.edgecolor'] = 'black'
    plt.rcParams['axes.linewidth'] = 0.8
    plt.rcParams['lines.linewidth'] = 1.5
    plt.rcParams['figure.titleweight'] = 'bold'
setmyplotstyle()

In [3]:
def format_float(value):
    return f'{value:,.2f}'
pd.options.display.float_format = format_float

In [4]:
# functions to clean up dataframe
def strip_clean_drop(dataframe):
    dataframe.columns = dataframe.columns.str.strip()  # gets rid of extra spaces
    dataframe.columns = dataframe.columns.str.lower()  # converts to lower case
    dataframe.columns = dataframe.columns.str.replace(' ', '_')  # allow dot notation with no spaces
    dataframe.columns = dataframe.columns.str.replace("'", '')  # allow dot notation with no special chars
    dataframe.columns = dataframe.columns.str.replace(".", '', regex=False)  # allow dot notation with no special chars
    dataframe = dataframe.dropna(axis=0, how="all")
    dataframe = dataframe.dropna(axis=1, how="all")
    dataframe = dataframe.convert_dtypes(convert_string=True)
    
    try:
        dataframe = dataframe.apply(lambda x: x.str.strip().lower() if x.dtypes=="string" else x)
    except:
        pass
    
    return dataframe

In [5]:
def getToken():
    # get token
    api_url = "https://www.ura.gov.sg/uraDataService/insertNewToken.action"
    headers = {
        'AccessKey':'32417271-1c2e-4a9f-bd17-f97c6ce79c4e',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:82.0) Gecko/20100101 Firefox/82.0'
    }
    response = requests.get(url=api_url, headers=headers)

    if response.status_code == 200:
        data = json.loads(response.text)
        dailyToken = data['Result']
        #print(response.headers)
        #print(response.text)
    else:
        print(f"Error: {response.status_code}")
        exit()
    
    return dailyToken

TOKEN = getToken()


In [6]:
def loadDataBatch(period):
    api_url = "https://www.ura.gov.sg/uraDataService/invokeUraDS"
    params = {'service': 'PMI_Resi_Rental', 'refPeriod':period}
    headers = {
        'AccessKey':'32417271-1c2e-4a9f-bd17-f97c6ce79c4e',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:82.0) Gecko/20100101 Firefox/82.0',
        'Token':TOKEN
    }
    
    response = requests.get(url=api_url, params=params, headers=headers)
    data = json.loads(response.text)
    data = data['Result']
    df = pd.json_normalize(data,'rental',['project','street'])
    
    return df

In [7]:
def getData():
    period_list = ["22q1","22q2","22q3","22q4","23q1"]
    dflist = []
    for count, period in enumerate(period_list):
        print(period)
        df = loadDataBatch(period)
        dflist.append(df)
    c_data = pd.concat(dflist)
    return c_data

In [8]:
df = getData()

22q1
22q2
22q3
22q4
23q1


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 107342 entries, 0 to 13682
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   areaSqm       107342 non-null  object
 1   leaseDate     107342 non-null  object
 2   propertyType  107342 non-null  object
 3   district      107342 non-null  object
 4   areaSqft      107342 non-null  object
 5   noOfBedRoom   107342 non-null  object
 6   rent          107342 non-null  int64 
 7   project       107342 non-null  object
 8   street        107342 non-null  object
dtypes: int64(1), object(8)
memory usage: 8.2+ MB


In [11]:
# clean up
pdata = (
    df.clean_names()
    .convert_dtypes()
    .change_type(['areasqm','district','areasqft','noofbedroom','rent'],dtype='float',ignore_exception='keep_values')
    .change_type(['propertytype','project','street'],dtype='string')
    .transform_columns(['propertytype','project','street'],lambda x: x.lower(),elementwise=True)
)

# transformations
pdata = (
    pdata#.add_column(column_name='rentpersqft',value=(pdata['rent']/pdata['areasqft']))
    .add_column(column_name='mth',value=pdata['leasedate'].str[0:2])
    .add_column(column_name='year',value=pdata['leasedate'].str[2:4])
)

pdata = pdata.add_column(column_name='_date',value=("01"+"/"+pdata["mth"]+"/"+pdata["year"]))
pdata = pdata.transform_columns(['_date'], lambda x: datetime.strptime(x, "%d/%m/%y").date(), elementwise=True)
pdata = pdata.change_type(['mth','year'],dtype='int',ignore_exception='keep_values')
pdata["_date"] = pd.to_datetime(pdata["_date"], dayfirst=True, errors="coerce")
pdata = pdata.sort_values('_date')


In [12]:
pdata.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 107342 entries, 0 to 4966
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   areasqm       107342 non-null  string        
 1   leasedate     107342 non-null  string        
 2   propertytype  107342 non-null  object        
 3   district      107342 non-null  float64       
 4   areasqft      107342 non-null  string        
 5   noofbedroom   107342 non-null  string        
 6   rent          107342 non-null  float64       
 7   project       107342 non-null  object        
 8   street        107342 non-null  object        
 9   mth           107342 non-null  int64         
 10  year          107342 non-null  int64         
 11  _date         107342 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(2), object(3), string(4)
memory usage: 10.6+ MB


In [ ]:
pdata.query('project.str.contains("irwell hill") & area==46',engine='python').groupby(['sqft','floorrange','_date']).agg(
    _count=('noofunits',np.sum),
    _meanprice_sqft=('pricepersqft',np.mean),
    _maxprice_sqft=('pricepersqft',np.max),
    _maxgrossprice=('price',np.max)
)

In [13]:
pdata.query("typeofsale==1 & area==46").groupby(['marketsegment','_date']).agg(
    _count=('noofunits',np.sum),
    _meanprice_sqft=('pricepersqft',np.mean),
    _maxprice_sqft=('pricepersqft',np.max),
    _maxgrossprice=('price',np.max)
)

_count  _meanprice_sqft  _maxprice_sqft  \
marketsegment _date                                                 
ccr           2019-01-01    1.00         2,474.03        2,474.03   
              2019-06-01    1.00         2,801.21        2,801.21   
              2019-09-01    1.00         2,377.09        2,377.09   
              2019-12-01    1.00         2,801.21        2,801.21   
              2020-01-01    3.00         2,928.89        3,019.32   
...                          ...              ...             ...   
rcr           2022-11-01    2.00         2,654.49        2,667.83   
              2022-12-01    1.00         2,786.84        2,786.84   
              2023-01-01    1.00         2,847.25        2,847.25   
              2023-02-01    1.00         2,650.75        2,650.75   
              2023-03-01    6.00         2,664.64        2,673.78   

                          _maxgrossprice  
marketsegment _date                       
ccr           2019-01-01    1,225,000.00  
              2019-06-01    1,387,000.00  
              2019-09-01    1,177,000.00  
              2019-12-01    1,387,000.00  
              2020-01-01    1,495,000.00  
...                                  ...  
rcr           2022-11-01    1,320,960.00  
              2022-12-01    1,379,888.00  
              2023-01-01    1,409,800.00  
              2023-02-01    1,312,502.00  
              2023-03-01    1,323,907.00  

[89 rows x 4 columns]

In [48]:
pdata.query('project.str.contains("euhabitat") & sqft==527 & year>=18').groupby(['floorrange','sqft','_date','typeofsale']).agg(
    _count=('noofunits',np.sum),
    _meanprice_sqft=('pricepersqft',np.mean),
    _maxprice_sqft=('pricepersqft',np.max),
    _maxgrossprice=('price',np.max)
)._maxprice_sqft.plot.bar()

UndefinedVariableError: name 'sqft' is not defined

In [35]:
g1 = pdata.query('project.str.contains("euhabitat") & year>=18 & areasqft=="500-600"').groupby(['_date']).agg(
    _meanRent=('rent',np.mean),
    _minRent=('rent',np.min),
    _maxRent=('rent',np.max)
)
g1

,_meanRent,_minRent,_maxRent
_date,,,
2022-01-01,"2,181.58","1,700.00","2,700.00"
2022-02-01,"2,216.67","1,800.00","2,700.00"
2022-03-01,"2,347.37","2,100.00","3,100.00"
2022-04-01,"2,184.75","1,850.00","2,800.00"
2022-05-01,"2,440.16","2,000.00","3,000.00"
2022-06-01,"2,436.36","1,950.00","3,000.00"
2022-07-01,"2,438.71","1,700.00","3,000.00"
2022-08-01,"2,588.89","2,000.00","3,400.00"
2022-09-01,"2,765.00","2,500.00","3,500.00"


In [36]:
fig = px.bar(g1, y=["_minRent","_meanRent","_maxRent"], barmode="group", text_auto=True)
fig.show()

In [13]:
ccr_d = [9,10,11]
g2 = pdata.query('district in @ccr_d & year>=18 & areasqft=="500-600"').groupby(['_date']).agg(
    _meanRent=('rent',np.mean),
    _minRent=('rent',np.min),
    _maxRent=('rent',np.max)
)
g2
fig = px.bar(g2, y=["_minRent","_meanRent","_maxRent"], barmode="group", text_auto=True)
fig.show()